# Predoc Data Task 3

*Gavin Qu*



## Overview

To render: run `quarto render analysis.qmd`
Additional documentations for reproducibility see **README.md**

## Setup



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import pyfixest as pf

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style='whitegrid', palette='colorblind')

**Discussion:**

*[Add discussion of the result above.]*




## Data Ingestion and Loading



In [ ]:
df = pd.read_csv("./data/cps_women_lfp.csv", low_memory=False)
df.info()
df.head(10)



**Note on columns:** 
- 'education' value are open sets
- 'college' is binary string (should be converted for memory efficiency)
- 'age' is binned, e.g. <25
- significant (40%+) missing value in income columns, is the missingness structurally correlated to another column? 
- 'lfp' and 'employed' can be binary flag as well
- 90%+ missing for 'covid_*' and 'telework_*' columns

## Variable Inventory



In [ ]:
for col in df.select_dtypes(include="str").columns:
    print(df[col].value_counts(dropna=False))



Note on income: CPS income data comes from the March supplement; basic monthly CPS doesn't have income

Col name map: {inctot: total personal income, incss: social security income, }

## Year Coverage & Sample Size
- Verify years 1994-2024, check for sample size


In [ ]:
df.groupby("year").size().plot(kind="bar", title="Observations per Year")
plt.ylabel("N")
plt.tight_layout()
plt.show()



## Numeric Variable 
- Check income variables for top-coding and missing values by column year


In [ ]:
print(df[["inctot", "incss", "income", "wgt"]].describe())

df.groupby("year")[["inctot", "incss", "income"]].apply(
    lambda x: x.isna().mean())



The missingness distribution by year between income variables are the same. The fact that missingness is identical across all three income variables across years suggests these are all coming from the ASEC supplement together. So missingness is structural — non-March rows have all three missing simultaneously. 

- 'wgt' is person-level weight from the CPS? 



In [ ]:
# wgt diagnostics to determine whether it's ASEC march data or monthly
df.groupby(df["income"].notna())["wgt"].describe()



**Note:** since wgt is evenly distributed across income missing and present rows, we can infer it's from monthly data. also known as 'WTFINL'. 

## Telework Variable Availability



In [ ]:
# These should only be populated from ~2020 onward
telework_cols = ["covid_telework", "covid_unemployed", "covid_paid",
                 "telework_now", "telework_before", "telework_difference"]
for col in telework_cols:
    print(f"\n {col} by year ")
    print(df.groupby("year")[col].apply(lambda x: (x.notna() & (x != "NIU")).mean()))



## Flags & Data Cleaning

**TODO: Document issues**
- [ ] Income top-code jumps?
- [x] Survey weight variable: is this ASEC or basic monthly? Monthly
- [ ] LFP / employed / self_employed: confirm coding (Yes/No? 1/0? labels?)
- [x] Age: is it numeric stored as string? Any "Under 1" type codes?

```# Convert types as needed
# df["age_num"] = ...
# df["lfp_bin"] = ...
```

---

# Context

The file cps women lfp.csv1 (also available as a .dta file) contains individual-level data from the U.S. Current Population Survey since 1994. The data set includes information on demographics and labor market outcomes. Please use the provided data to answer each
question below in complete sentences. Please submit a pdf of your written responses and any graphs and/or tables that support your responses, as well as your code. You can use any programming language of your choice, though you may find the workshop more beneficial if you use R or STATA.

# Part 1: Labor Force Participation

*Budget: ~4 hours*

## Q1: Overall Trends in Female LFP Since 1994

How has female labor force participation evolved since 1994? Please provide graphs and/or tables to support your answer. 

**Approach:** Compute weighted annual LFP rates for all women, plot the time series, and narrate key inflection points (late-1990s plateau, post-2000 decline, COVID shock, recovery).

#### Global setting for colorblind friendly plots 


In [ ]:
# color blind friendly theme for all plots
sns.set_theme(style='whitegrid', palette='colorblind')

**Discussion:**

*[Add discussion of the result above.]*




#### Helper function 



In [ ]:
# helper function for applying weights and weighted means
def weighted_mean(data, value_col, weight_col):
    valid = data[[value_col, weight_col]].dropna()

    if valid.empty:
        return np.nan

    return np.average(valid[value_col], weights=valid[weight_col])

def wmean(values, weights):
    mask = values.notna() & weights.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

**Discussion:**

*[Add discussion of the result above.]*




#### Subset and group by year 


In [ ]:
df_women = df.loc[df['sex'] == 'Female'].copy()
df_women['lfp_bin'] = (df_women['lfp'] == 'In labor force').astype(int)

# weight LFP rate by year
lfp_by_year = (
    df_women.groupby('year')
    .apply(lambda g: wmean(g['lfp_bin'], g['wgt']))
    .rename('lfp_rate')
)



Note: LFP-missing observations constitutes <1% of the female sample; results are insensitive to these.

#### Plotting



In [ ]:
import matplotlib.ticker as mticker
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(lfp_by_year.index, lfp_by_year.values,
        marker='o', markersize=4, linewidth=1.8,
        color='#0127b2')

# recession shading
ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

# Annotations with labels
peak_year = lfp_by_year.idxmax()
peak_val = lfp_by_year.max()
ax.annotate(f'Peak: {peak_val:.1%}\n({peak_year})',
            xy=(peak_year, peak_val),
            xytext=(peak_year - 3, peak_val + 0.005),
            fontsize=9, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', alpha=0.6))

covid_val = lfp_by_year.loc[2020]
ax.annotate('COVID-19',
            xy=(2020, covid_val),
            xytext=(2016, covid_val - 0.015),
            fontsize=9, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', alpha=0.6))

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Labor Force Participation Rate')
ax.set_title('Female Labor Force Participation, 1994–2024', loc='left',
             fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



**Discussion:**


## Q2: Heterogeneity in LFP Changes (Women > 25)

Among women older than 25, which groups (race, age, income percentile, etc.) of
people had the biggest changes in labor force participation since 1994? Please provide at least three graphs and/or tables to support your answer.

Note: income quantile is partially endogenous to labor force status — non-participants tend to have lower individual income. I use income_quantiles (total income) rather than wageinc_quantiles (wage income) since the former is less mechanically tied to LFP, but causal interpretation should still be cautious.

### By Race



In [ ]:
cols = ['year', 'lfp', 'wgt', 'race', 'age', 'income_quantiles',
        'education', 'college', 'self_employed', 'inctot', 'incss', 'income']
df_women = df.loc[df['sex'] == 'Female', cols].copy()
df_women['lfp_bin'] = (df_women['lfp'] == 'In labor force').astype(int)

df_w25 = df_women.loc[df_women['age'] != '< 25'].copy()

# helper for weighted LFP by year × group
def lfp_by_year_group(data, group_col):
    return (data.groupby(['year', group_col], observed=True)
                .apply(lambda g: wmean(g['lfp_bin'], g['wgt']))
                .unstack(group_col))



*Note:* I will exclude "Native American" and "Two or more" from the race plot due to small sample sizes, which produces noisy year-to-year estimates. These groups remain in the summary table for completeness.



In [ ]:
race_keep = ['White', 'Black', 'Hispanic', 'Asian or Pacific Islander']
lfp_race = lfp_by_year_group(
    df_w25.loc[df_w25['race'].isin(race_keep)],
    'race'
)

fig, ax = plt.subplots(figsize=(9, 5.5))
lfp_race.plot(ax=ax, marker='o', markersize=3, linewidth=1.5)

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Labor Force Participation Rate')
ax.set_title('Female LFP by Race, Women 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.legend(title='Race', frameon=False, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*




### By Age Group



In [ ]:
# Set categorical order so the legend is sorted by age
age_order = ['25-34', '35-44', '45-54', '55-64', '65-74', '75+']
df_w25['age'] = pd.Categorical(df_w25['age'], categories=age_order, ordered=True)

lfp_age = lfp_by_year_group(df_w25, 'age')

fig, ax = plt.subplots(figsize=(9, 5.5))
lfp_age.plot(ax=ax, marker='o', markersize=3, linewidth=1.5)

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Labor Force Participation Rate')
ax.set_title('Female LFP by Age Group, Women 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.legend(title='Age', frameon=False, loc='center left', bbox_to_anchor=(1, 0.5))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*




### By Income Quantile



In [ ]:
# Income quantiles only populated for ASEC rows
income_order = ['0-19.99', '20-39.99', '40-59.99', '60-79.99', '80-100']
df_w25_inc = df_w25.dropna(subset=['income_quantiles']).copy()
df_w25_inc['income_quantiles'] = pd.Categorical(
    df_w25_inc['income_quantiles'], categories=income_order, ordered=True
)

lfp_inc = lfp_by_year_group(df_w25_inc, 'income_quantiles')

fig, ax = plt.subplots(figsize=(9, 5.5))
lfp_inc.plot(ax=ax, marker='o', markersize=3, linewidth=1.5)

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Labor Force Participation Rate')
ax.set_title('Female LFP by Income Quintile, Women 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.legend(title='Income Quintile', frameon=False, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
def endpoint_change(series, start=(1994, 1995, 1996), end=(2022, 2023)):
    # .reindex() is "safe" - it returns NaN for missing years instead of crashing
    start_vals = series.reindex(list(start))
    end_vals = series.reindex(list(end))
    
    start_mean = start_vals.mean()
    end_mean = end_vals.mean()
    
    return pd.Series({
        '1994-96': start_mean,
        '2022-23': end_mean,
        'Change (pp)': (end_mean - start_mean) * 100
    })

# The rest of your logic remains the same
summary = pd.concat([
    lfp_race.apply(endpoint_change).T.assign(Dimension='Race'),
    lfp_age.apply(endpoint_change).T.assign(Dimension='Age'),
    lfp_inc.apply(endpoint_change).T.assign(Dimension='Income Quintile'),
]).reset_index().rename(columns={'index': 'Group'})

summary = summary[['Dimension', 'Group', '1994-96', '2022-23', 'Change (pp)']]
display(summary)


**Discussion:**

*[Which groups changed most? Is the story about convergence (gaps closing)
or divergence? Any group that bucks the overall trend?]*

Race has the largest heterogeneity, of the four categories included, white female trends down slightly in LFP, where other groups seems to converge after the great recession. Since the covid-19 pandemic is still a relatively recent shock, I want to caution against over-extrapolating the pattern but it seems to be fit the same narrative. 

## Q3: Trends in Wages, Social Insurance, and Education (Women > 25)
Use the data to examine trends among women older than 25 for each of the following factors from 1994 to 2024. 
### (a) Wage and Salary Income

Note: INCTOT, INCWAGE reflect the dollar value at the time the survey was taken, not adjusted for inflation.



In [ ]:
# CPI-U deflator, 2024 base — VERIFY against BLS before final submission
cpi = {
    1994: 39.4, 1995: 40.5, 1996: 41.7, 1997: 42.6, 1998: 43.3,
    1999: 44.3, 2000: 45.7, 2001: 47.0, 2002: 47.8, 2003: 48.8,
    2004: 50.1, 2005: 51.8, 2006: 53.5, 2007: 55.0, 2008: 57.1,
    2009: 56.9, 2010: 57.8, 2011: 59.7, 2012: 60.9, 2013: 61.8,
    2014: 62.8, 2015: 62.9, 2016: 63.7, 2017: 65.0, 2018: 66.6,
    2019: 67.8, 2020: 68.7, 2021: 71.9, 2022: 77.7, 2023: 80.9,
    2024: 83.5
}
deflator = pd.Series({y: cpi[2024]/v for y, v in cpi.items()})

# Apply to ASEC subsample (income variables only populated in ASEC)
df_w25_inc = df_w25.dropna(subset=['income']).copy()
df_w25_inc['income_real']  = df_w25_inc['income']  * df_w25_inc['year'].map(deflator)
df_w25_inc['incss_real']   = df_w25_inc['incss']   * df_w25_inc['year'].map(deflator)

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
# quick sanity check 
df[['year']].describe()
df_w25.loc[df_w25['inctot'].notna(), ['inctot', 'incss', 'income']].head(20)

# np.average doesn't have a weighted-median equivalent
def wmedian(values, weights):
    mask = values.notna() & weights.notna()
    if mask.sum() == 0:
        return np.nan
    v, w = values[mask].values, weights[mask].values
    order = np.argsort(v)
    v, w = v[order], w[order]
    cum = np.cumsum(w)
    return v[np.searchsorted(cum, cum[-1] / 2)]

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
# Restrict to women with positive wage income (workers)
df_w25_workers = df_w25_inc.loc[df_w25_inc['income'] > 0].copy()

wage_by_year = (df_w25_workers.groupby('year')
                .apply(lambda g: wmedian(g['income_real'], g['wgt']))
                .rename('median_wage_real'))

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(wage_by_year.index, wage_by_year.values,
        marker='o', markersize=4, linewidth=1.8, color='#0127b2')

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_xlabel('Year')
ax.set_ylabel('Median Wage and Salary Income (2024 dollars)')
ax.set_title('Female Wage and Salary Income, Workers Age 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



Note: I report median wage and salary income conditional on positive earnings (workers only). This isolates earnings dynamics from changes in labor force composition.

### (b) Social Insurance Income

Note: I use incss, which captures Social Security income; broader social insurance categories such as SSI are not separately available in this extract.



In [ ]:
# Extensive margin: share with any SS income > 0
df_w25_inc['has_ss'] = (df_w25_inc['incss'] > 0).astype(int)
ss_share = (df_w25_inc.groupby('year')
            .apply(lambda g: wmean(g['has_ss'], g['wgt']))
            .rename('share_with_ss'))

# Intensive margin: median SS income among recipients (real)
ss_recipients = df_w25_inc.loc[df_w25_inc['incss'] > 0]
ss_median = (ss_recipients.groupby('year')
             .apply(lambda g: wmedian(g['incss_real'], g['wgt']))
             .rename('median_ss_real'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: extensive margin
axes[0].plot(ss_share.index, ss_share.values,
             marker='o', markersize=4, linewidth=1.8, color='#0127b2')
axes[0].axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
axes[0].axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Share Receiving Social Security')
axes[0].set_title('Extensive Margin', loc='left', fontsize=11, fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(True, alpha=0.3)

# Panel 2: intensive margin
axes[1].plot(ss_median.index, ss_median.values,
             marker='o', markersize=4, linewidth=1.8, color='#DE8F05')
axes[1].axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
axes[1].axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median SS Income, Recipients (2024 dollars)')
axes[1].set_title('Intensive Margin', loc='left', fontsize=11, fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Social Security Income, Women 25+', fontsize=13, fontweight='bold',
             x=0.02, ha='left', y=1.02)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*




### (c) Education Attainment



In [ ]:
edu_order = ["< HS Diploma", "HS Diploma", "Some college, no degree",
             "Associate's Degree", "Bachelor's Degree", "Master's or Higher"]

df_w25['education'] = pd.Categorical(df_w25['education'],
                                      categories=edu_order, ordered=True)

# Weighted share in each education category by year
def edu_share_by_year(data):
    out = (data.groupby(['year', 'education'], observed=True)
              .apply(lambda g: g['wgt'].sum())
              .unstack('education', fill_value=0))
    return out.div(out.sum(axis=1), axis=0)

edu_shares = edu_share_by_year(df_w25)

fig, ax = plt.subplots(figsize=(9, 5.5))
edu_shares.plot.area(ax=ax, alpha=0.8, linewidth=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Share of Women 25+')
ax.set_title('Education Attainment, Women 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_xlim(edu_shares.index.min(), edu_shares.index.max())
ax.legend(title='Education', frameon=False, loc='center left',
          bbox_to_anchor=(1, 0.5), reverse=True)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()



**Discussion:**

Based on these trends, what factors could be driving the patterns you found in Questions 1 and 2?

Discussion: The pattern to highlight here is how female wage and income have consistently been trending down after the great recession, bigger drops tend to happen after major systematic shocks. I decomposed total social income change to the following: extensive margin increase reflect the growing number of women near retirement age or retired, where decreasing intensive margin imply the median amount is decreasing. 
In terms of education amongst women 25+, HS diploma trends down slightly, which can be explained by the increase in those with Bachelor's degree and up. 
Extensive Margin: total share of employement taken into account.
Intensive Margin: Only counts those receiving social security. 

## Q4: Steepest Year-over-Year Increase in Female LFP

Between 1994 and 2024, which year had the steepest increase in female labor force participation relative to the previous year? What factors do you think are driving this pattern? Support your answers by using the data, referencing major events that happened around this time period, and/or citing previous studies.



In [ ]:
yoy_change = lfp_by_year.diff()
peak_year = yoy_change.idxmax()
peak_change = yoy_change.max()

print(f"Steepest YoY increase: {peak_year}")
print(f"Change: {peak_change*100:.2f} percentage points")
print(f"LFP {peak_year-1}: {lfp_by_year[peak_year-1]:.1%}")
print(f"LFP {peak_year}: {lfp_by_year[peak_year]:.1%}")

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#DE8F05' if y == peak_year else '#0173B2' for y in yoy_change.index]
ax.bar(yoy_change.index, yoy_change.values * 100, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('YoY Change in LFP (percentage points)')
ax.set_title('Year-over-Year Change in Female LFP, 1995–2024', loc='left',
             fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()



Discussion: The data points to 1997 as the steepest YoY increase with 1.08 percentage point increase. Congress passed the Personal Responsibility and Work Opportunity Reconciliation Act in 1996, which added work requirements for single mothers receiving transfer payments. I beleive this would potentially contribute to the jump prima facie. Such policy changes would appear as shocks to the labor supply for the female population, as demonstrated by Meyer and Rosenbaum in their 2001 study. 


## Q5: LFP by College Education

How has labor force participation for college-educated and not college-educated women evolved since 1994? Please provide graphs and/or tables to support your answer. 



In [ ]:
lfp_college = lfp_by_year_group(df_w25, 'college')

fig, ax = plt.subplots(figsize=(9, 5.5))
lfp_college.plot(ax=ax, marker='o', markersize=4, linewidth=1.8)

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Labor Force Participation Rate')
ax.set_title('Female LFP by College Education, Women 25+', loc='left',
             fontsize=12, fontweight='bold')
ax.legend(title='Education', frameon=False, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

endpoint_change_table = lfp_college.apply(endpoint_change).T
endpoint_change_table



I only included those older than 25, becaue 22 is the normal age of college graduates and age of first job post-college can depend on labor market conditions and business cycles. 
As shown on the graph, female labor force participation for those over the age of 25 seems to be decreasing over all, notably after the great recession and covid-19 pandemic. However, labor force particpation rate of college degree holders is significantly higher than those without. 
College-educated women also suffered a smaller LFP drop in 2020. 


## Q6: Alternative LFP Measure (Excluding Self-Employed)

Create an alternative measure of labor force participation that excludes individuals from the labor force if they are self-employed in their main job (lfp = 0 if self-employed in main job). Using the new measure, describe how labor force participation for college-educated and not college-educated women has evolved since 1994. Please provide graphs and/or tables to support your answer.

note: Unemployed labor force participants (lfp = In labor force, self_employed = NaN) are coded as lfp_alt = 1 since the self-employment question applies only to current jobs.



In [ ]:
# verify self employment flag for female subset
df_w25['self_employed'].value_counts(dropna=False)

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
df_w25['lfp_alt'] = np.where(
    (df_w25['lfp'] == 'In labor force') & (df_w25['self_employed'] != 'Self-employed'),
    1, 0
)

lfp_alt_college = (df_w25.groupby(['year', 'college'], observed=True)
                   .apply(lambda g: wmean(g['lfp_alt'], g['wgt']))
                   .unstack('college'))

fig, ax = plt.subplots(figsize=(9, 5.5))
lfp_alt_college.plot(ax=ax, marker='o', markersize=4, linewidth=1.8)

ax.axvspan(2007, 2009, alpha=0.12, color='gray', zorder=0)
ax.axvspan(2020, 2020.5, alpha=0.12, color='gray', zorder=0)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Alternative LFP Rate (excl. self-employed)')
ax.set_title('Alternative Female LFP by College Education, Women 25+',
             loc='left', fontsize=12, fontweight='bold')
ax.legend(title='Education', frameon=False, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
se_rate = (df_w25.loc[df_w25['lfp_bin'] == 1]
           .groupby(['year', 'college'], observed=True)
           .apply(lambda g: wmean((g['self_employed'] == 'Self-employed').astype(int), g['wgt']))
           .unstack('college'))
print(se_rate.loc[[1994, 2024]])
print("\nMean self-employment rate by college status:")
print(se_rate.mean())



**Discussion:**
The gap between the two groups becomes narrower modestly when self-employment is excluded, this concludes that self-employment rates among labor force participants are nearly identical across education groups — 9% for college-educated women and 8% for non-college women on average. This seem to imply that more women are self-employed in the college-educated group. 

## Q7: Comparing the Two LFP Measures

How does our labor market analysis change when we use the new measure? Which
measure do you prefer? Explain

Discussion: Our labor market analysis does not change much, since both lines shifted down by 8-9 percent points, neither does the trajectory. I prefer the first measure including the self-employed group, e.g. a self-employed accountant or consultant by many characteristics should be no different than a salaried one. 

The research question should also dictate my answer to the question: if our research question is to study employer-provided benefit and treatment associated with a specific type of employment, then excluding the self-employed group makes sense. However, if we want to study the effect of work in general, we should include those self-employed data. 


---

# Part 2: Telework

*Budget: ~2 hours*

## Q1: Outcomes by Telework Status (2020–2024)

Since the rise of telework in 2020, how have wages, employment, and labor force participation changed for women who had telework from 2020-2024 and women who did not? Please provide at least three graphs and/or tables to support your answer. 

**Approach:** I'm using 'telework_now': which is the number of women who had teleworked in the past week, with 84K non-null. 




In [ ]:
for col in ['telework_now', 'covid_telework', 'telework_before']:
    print(f"\n{col} by year:")
    print(df.groupby('year')[col].apply(lambda x: x.notna().mean()).round(3))

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
df_women = df[df['sex'] == 'Female'].copy()

# Subset: women in years where telework is observed
df_tw = df_women.loc[df_women['telework_now'].notna()].copy()
df_tw['teleworked'] = (df_tw['telework_now'] == 'Had telework in the past week').astype(int)
df_tw['employed_bin'] = (df_tw['employed'] == 'Employed').astype(int)
df_tw['lfp_bin'] = (df_tw['lfp'] == 'In labor force').astype(int)

print(df_tw.groupby('year')['teleworked'].agg(['mean', 'size']))

**Discussion:**

*[Add discussion of the result above.]*




### Wages by Telework Status





Note: Wage comparisons by telework status are not feasible in this extract: telework variables (telework_now) are populated in 2023–2024 basic monthly CPS, while income variables come from the ASEC supplement, with no overlapping observations. The employment and LFP comparisons below use the basic monthly sample where telework is observed. 

### Employment by Telework Status



In [ ]:
emp_tw = (df_tw.groupby(['year', 'teleworked'])
          .apply(lambda g: wmean(g['employed_bin'], g['wgt']))
          .unstack('teleworked')
          .rename(columns={0: 'No telework', 1: 'Telework'}))

fig, ax = plt.subplots(figsize=(9, 5))
emp_tw.plot(ax=ax, marker='o', markersize=5, linewidth=1.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('Employment Rate')
ax.set_title('Employment Rate by Telework Status', loc='left', fontsize=12, fontweight='bold')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*




### LFP by Telework Status



In [ ]:
lfp_tw = (df_tw.groupby(['year', 'teleworked'])
          .apply(lambda g: wmean(g['lfp_bin'], g['wgt']))
          .unstack('teleworked')
          .rename(columns={0: 'No telework', 1: 'Telework'}))

fig, ax = plt.subplots(figsize=(9, 5))
lfp_tw.plot(ax=ax, marker='o', markersize=5, linewidth=1.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Year')
ax.set_ylabel('LFP Rate')
ax.set_title('LFP Rate by Telework Status', loc='left', fontsize=12, fontweight='bold')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



**Discussion:** Teleworkers have higher LFP (selection into telework-capable jobs, since those are likely all office-jobs). Teleworkers have higher employment and LFP rates than non-teleworkers in 2023–2024. These gaps are descriptive: telework availability is correlated with occupation, education, and industry, so we are comparing groups that differ on many observables. The data are consistent with telework being a privilege of higher-attachment, higher-skill workers rather than a treatment with a clean causal effect on labor market outcomes.


## Q2: Who Teleworked During the Pandemic? (Women > 25, 2021)

For which groups of women older than 25 was telework due to the pandemic most
common in 2021? Based on these patterns, what can you infer about the relationship between economic well-being and the ability to telework between 2021? Please provide at least three graphs and/or tables to support your answer.

Note from BLS website: "BLS added new questions to the Current Population Survey (CPS) starting in October 2022 that focus on telework or work at home for pay."



In [ ]:
# Subset: women 25+, 2021, with covid_telework observed
df_2021 = df.loc[
    (df['sex'] == 'Female') 
    & (df['age'] != '< 25') 
    & (df['year'] == 2021) 
    & (df['covid_telework'].notna())
].copy()
df_2021['tw_pandemic'] = (df_2021['covid_telework'] == 
                          'Telework from 2021-2022 due to COVID').astype(int)
print(f"N: {len(df_2021)}")

**Discussion:**

*[Add discussion of the result above.]*




### Telework by Race



In [ ]:
race_keep = ['White', 'Black', 'Hispanic', 'Asian or Pacific Islander']
tw_race = (df_2021.loc[df_2021['race'].isin(race_keep)]
           .groupby('race', observed=True)
           .apply(lambda g: wmean(g['tw_pandemic'], g['wgt']))
           .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
tw_race.plot(kind='barh', ax=ax, color='#0122b2')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Pandemic Telework Rate')
ax.set_ylabel('')
ax.set_title('Pandemic Telework Rate by Race, Women 25+ (2021)',
             loc='left', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**Discussion:**

*[Add discussion of the result above.]*





### Telework by Education



In [ ]:
tw_college = (df_2021.groupby('college', observed=True)
              .apply(lambda g: wmean(g['tw_pandemic'], g['wgt']))
              .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
tw_college.plot(kind='barh', ax=ax, color='#b605de')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel('Pandemic Telework Rate')
ax.set_ylabel('')
ax.set_title('Pandemic Telework Rate by College Education, Women 25+ (2021)',
             loc='left', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()



**Discussion:**

## Q3: Counterfactual — What If Telework Wasn't an Option?

Predict what trends in wages, employment, and labor force participation for college-educated women from 2020 to 2024 would have looked like if telework was not an option. What does this tell you about the economic impacts of telework during the COVID-19 pandemic? Please support your answer with graphs and/or tables. 

**Approach:** Y = labor market outcome, T = telework, use non-college educated women as counterfactual in 2x2 DiD and 2010-2019 as a pre-treatment period. 
$$τ^=(Ycollege, post​−Ycollege, pre​)−(Ynon, post​−Ynon, pre​)$$

Assumption: Paraellel Trend, need to inspect visually at least. 



In [ ]:
cols_needed = ['year', 'sex', 'age', 'lfp', 'employed', 'college', 
               'self_employed', 'race', 'wgt', 'covid_telework', 'telework_now']

df_women = df.loc[df['sex'] == 'Female', cols_needed].copy()
df_women['lfp_bin'] = (df_women['lfp'] == 'In labor force').astype(int)
df_women['employed_bin'] = (df_women['employed'] == 'Employed').astype(int)

df_w25 = df_women.loc[df_women['age'] != '< 25'].copy()

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
# Build employment series by year × college, women 25+
emp_college = (df_w25.groupby(['year', 'college'], observed=True)
               .apply(lambda g: wmean(g['employed_bin'], g['wgt']))
               .unstack('college'))

# Rebuild LFP series too if not in scope
lfp_college = (df_w25.groupby(['year', 'college'], observed=True)
               .apply(lambda g: wmean(g['lfp_bin'], g['wgt']))
               .unstack('college'))

def did_counterfactual(series_wide, treated_col, control_col, base_year=2019, 
                       post_start=2020, post_end=2024):
    """DiD counterfactual: apply control's post-base deviations to treated baseline."""
    treated_base = series_wide.loc[base_year, treated_col]
    control_base = series_wide.loc[base_year, control_col]
    post_years = range(post_start, post_end + 1)
    return pd.Series(
        {y: treated_base + (series_wide.loc[y, control_col] - control_base) 
         for y in post_years},
        name='counterfactual'
    )

cf_emp = did_counterfactual(emp_college, 'Has college degree', 'No college degree')
cf_lfp = did_counterfactual(lfp_college, 'Has college degree', 'No college degree')

def plot_did(series_wide, cf, ylabel, title, treated_col='Has college degree',
             control_col='No college degree'):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    
    # Actuals — both groups, full range
    full = series_wide.loc[2010:2024]
    ax.plot(full.index, full[treated_col], marker='o', markersize=4, 
            linewidth=1.8, color='#6801b2', label='College (actual)')
    ax.plot(full.index, full[control_col], marker='s', markersize=4, 
            linewidth=1.8, color='#deba05', label='Non-college (actual)')
    
    # Counterfactual: anchor at 2019 then follow control deviations
    cf_full = pd.concat([pd.Series({2019: series_wide.loc[2019, treated_col]}), cf])
    ax.plot(cf_full.index, cf_full.values, marker='D', markersize=4, 
            linewidth=1.8, linestyle='--', color='#ea241d', 
            label='College (counterfactual)')
    
    # Shade the gap
    actual_post = series_wide.loc[2020:2024, treated_col]
    ax.fill_between(actual_post.index, actual_post.values, cf.values,
                    alpha=0.15, color='#ce4f4f', label='Implied differential effect')
    
    ax.axvline(2020, color='gray', linestyle=':', alpha=0.6, linewidth=1)
    
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc='left', fontsize=12, fontweight='bold')
    ax.legend(frameon=False, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_did(emp_college, cf_emp, 'Employment Rate',
         'Employment: Actual vs. DiD Counterfactual, College Women 25+')

plot_did(lfp_college, cf_lfp, 'LFP Rate',
         'LFP: Actual vs. DiD Counterfactual, College Women 25+')

**Discussion:**

*[Add discussion of the result above.]*


In [ ]:
def did_summary(series_wide, cf, treated_col='Has college degree'):
    actual = series_wide.loc[2020:2024, treated_col]
    return pd.DataFrame({
        'Actual': actual,
        'Counterfactual': cf,
        'Gap (pp)': (actual - cf) * 100
    })

print("Employment:")
print(did_summary(emp_college, cf_emp).round(3))
print("\nLFP:")
print(did_summary(lfp_college, cf_lfp).round(3))



**Discussion:**
The gaps are tiny (under 1 percentage point) and they go the wrong direction in 2020 and 2024 — college women fared worse than the counterfactual implies, not better. Three plausible interpretations:

1. Non-college women are a poor control group. They were hit much harder by COVID (service-sector job losses), so applying their trajectory as the counterfactual produces a low bar that college women easily clear. The DiD estimate is the differential effect — college relative to non-college — not the level effect of telework. If both groups benefited from telework but non-college benefited less, the DiD estimate would be small.

2. College-educated women's LFP/employment was already high and stable (~70%), with limited room to fall. The COVID shock fell more on hours worked, wages, and informal labor than on the binary employment-rate measure for this group.

3. Parallel trends violation in the wrong direction. If college women's pre-2020 LFP was decelerating relative to non-college (look at 2018-2019), the counterfactual extrapolation is biased. 

4. However, the counter factual post-covid has lower employment than the actual data. It is possible that telework affected the employment rebound rate after the pandemic among college-educated groups. 
